In [ ]:
from pathlib import Path

# define Olive generated OVIR ONNX Encapsulated model paths
encoder_path = Path("model/encoder/sam21_encoder_pt_ov_int8_quant_st.onnx")
decoder_path = Path("model/decoder/sam21_decoder_pt_ov_int8_quant_st.onnx")

# define EP and device
ExecutionProvider="OpenVINOExecutionProvider"
ov_device = "NPU"

In [ ]:
from winml import register_execution_providers
register_execution_providers()

In [ ]:
from urllib import request

test_image_url = "https://github.com/facebookresearch/segment-anything/blob/main/notebooks/images/truck.jpg?raw=true"
test_image_name = "truck.jpg"

request.urlretrieve(test_image_url, test_image_name)

from IPython.display import Image, display

display(Image(filename=test_image_name))

In [ ]:
# open the image in PIL
from PIL import Image as PILImage
input_image = PILImage.open(test_image_name)

# define the foreground, background and bbox for segmentation
foreground_points = [(500, 375), (1125, 625), (575, 750), (1405, 575)]
background_points = []
bbox = None

In [ ]:
import sys

NOTEBOOK_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from ov_inference_utils import FacebookSam2_1HieraSmall, IPromptedImageSegmentationModel

# create and instantiate the ORT model sessions for the vision encoder and mask decoder
model = FacebookSam2_1HieraSmall(
    model_path = {
        "vision_encoder": encoder_path,
        "mask_decoder": decoder_path
    },
    device=ov_device,
    execution_provider=ExecutionProvider
)

In [ ]:
# run forward pass with the input image and prompts to generate the prediction using the segment_image function of the model class
prediction = model.segment_image(
    image=input_image,
    prompt=IPromptedImageSegmentationModel.Prompt(
        bbox=bbox,
        foreground_points=foreground_points,
        background_points=background_points
    )
)

In [ ]:
# Plot the image + bbox + foreground and background points + predicted mask
import numpy as np
from IPython.display import display

# convert input PIL image to numpy array
image_array = np.array(input_image)

if prediction.mask is not None:
    mask_array = prediction.mask

    # Alpha-blend mask overlay onto image
    overlay = image_array.copy().astype(np.float32)
    mask_rgb = np.zeros_like(image_array, dtype=np.float32)
    mask_rgb[mask_array > 0] = [0, 100, 255]
    alpha = 0.75
    overlay[mask_array > 0] = (
        overlay[mask_array > 0] * (1 - alpha) + mask_rgb[mask_array > 0] * alpha
    )
    overlay = overlay.astype(np.uint8)

    # Draw bbox on overlay
    if bbox:
        x, y, w, h = bbox
        thickness = 2
        overlay[y:y+h, x:x+thickness] = [0, 0, 255]
        overlay[y:y+h, x+w-thickness:x+w] = [0, 0, 255]
        overlay[y:y+thickness, x:x+w] = [0, 0, 255]
        overlay[y+h-thickness:y+h, x:x+w] = [0, 0, 255]

    # Draw foreground points on overlay
    for pt in foreground_points:
        x, y = int(pt[0]), int(pt[1])
        rr, cc = np.ogrid[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)]
        mask_circle = (rr - y)**2 + (cc - x)**2 <= 36
        overlay[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)][mask_circle] = [0, 255, 0]

    # Draw background points on overlay
    for pt in background_points:
        x, y = int(pt[0]), int(pt[1])
        rr, cc = np.ogrid[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)]
        mask_circle = (rr - y)**2 + (cc - x)**2 <= 36
        overlay[max(0, y-6):min(overlay.shape[0], y+7), max(0, x-6):min(overlay.shape[1], x+7)][mask_circle] = [255, 0, 0]

    display(PILImage.fromarray(overlay))
else:
    print("No mask predicted.")
    display(PILImage.fromarray(image_array))